In [ ]:
%%capture
%pip install -U transformers 
%pip install -U accelerate 
%pip install -U peft 
%pip install -U trl 
%pip install -U bitsandbytes 
%pip install -U wandb

In [ ]:
# import trl; print(trl.__version__)
import trl
print([x for x in dir(trl) if 'chat' in x.lower()])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch

device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
print(device)

model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)
dataset = load_dataset("HuggingFaceTB/smoltalk", "all")

finetune_name = "SmolLM2-FT-MyDataset"
finetune_tags = ["smol-course", "module_1"]

# ── Replace setup_chat_format manually ──────────────────────────────────────
tokenizer.chat_template = """{% for message in messages %}{% if message['role'] == 'user' %}<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}<|im_start|>assistant
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'system' %}<|im_start|>system
{{ message['content'] }}<|im_end|>
{% endif %}{% endfor %}{% if add_generation_prompt %}<|im_start|>assistant
{% endif %}"""

tokenizer.add_special_tokens({
    "additional_special_tokens": ["<|im_start|>", "<|im_end|>"]
})
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<|im_end|>"})

model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id


# Generate with the base model

In [ ]:
prompt = "Write a haiku about programming"
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Configuring the SFTTrainer

In [ ]:
from huggingface_hub import login

login(token="")



In [ ]:
from transformers import TrainerCallback
from huggingface_hub import whoami

# ── Verify you're logged in ───────────────────────────────────────────────
print(whoami()["name"])  # will error if not logged in

class LossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"Step: {state.global_step} | Loss: {logs['loss']:.4f} | Eval Loss: {logs.get('eval_loss', 'N/A')}")

args = SFTConfig(
    output_dir="./sft_output",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    report_to="none",
    push_to_hub=True,
    hub_model_id="gugukaka/SmolLM2-FT-MyDataset",  
                          
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    callbacks=[LossCallback()],
)
trainer.train()

# ── Final push after training ─────────────────────────────────────────────
trainer.push_to_hub()